# Getting started with desdeo-brb

This notebook introduces Belief Rule-Based (BRB) systems and walks through a complete example: modeling the function $f(x) = x \sin(x^2)$ on $[0, 3]$, following the example in Chen et al. (2011), Section 4.2, Eq. 8.

The BRB inference methodology (RIMER) was introduced by Yang et al. (2006), and the adaptive training approach used here follows Chen et al. (2011).

```
# requires: pip install matplotlib
```


## What is a BRB system?

A Belief Rule-Based (BRB) system is a generalization of traditional IF-THEN rule bases. In a classical rule base, each rule maps crisp input conditions to a crisp output. In a BRB, each rule's consequent is instead a **belief distribution** over a set of possible output values. For example, rather than saying "if temperature is High, then risk is Medium," a BRB rule might say "if temperature is High, then risk is {Low: 0.1, Medium: 0.7, High: 0.2}."

Inference in a BRB is performed using the **Evidential Reasoning (ER)** algorithm, which analytically combines all activated rules into a single output distribution. This is more principled than simple weighted averaging and naturally handles incomplete or conflicting evidence.

A key advantage of BRB systems is that they are **both interpretable and trainable**. The initial rule base can come from expert knowledge (no data required), and the parameters can then be fine-tuned from data. Every intermediate quantity: which rules fired, how strongly, what the output distribution looks like. This is directly inspectable.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Example from Chen et al. (2011), Eq. 8
def f(x):
    return x * np.sin(x**2)


x_dense = np.linspace(0, 3, 500)
plt.figure(figsize=(8, 4))
plt.plot(x_dense, f(x_dense), "k-", linewidth=2)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title(r"$f(x) = x \sin(x^2)$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Referential values

A BRB system discretizes the input and output spaces using **referential values**: anchor points that define the "vocabulary" of the rules.

For this example (following the paper), we use:

- **7 precedent referential values** for $x$: $\{0, 0.5, 1, 1.5, 2, 2.5, 3\}$
- **5 consequent referential values** for $f(x)$: $\{-2.5, -1, 1, 2, 3\}$

The Cartesian product of precedent referential values gives 7 rules (one attribute, 7 values). Each rule's belief degrees are initialized by evaluating $f$ at the corresponding referential value and distributing the result as a belief over the consequent values.


In [ ]:
from desdeo_brb import BRBModel

# Referential values
prv = [np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])]
crv = np.array([-2.5, -1.0, 1.0, 2.0, 3.0])

# Construct model: initial beliefs computed from f at each referential value
model = BRBModel(
    prv,
    crv,
    initial_rule_fn=lambda x: x[0] * np.sin(x[0] ** 2),
)

print(f"Number of rules: {model.rule_base.n_rules}")
print(f"Number of attributes: {model.rule_base.n_attributes}")
print(f"Number of consequents: {model.rule_base.n_consequents}")

## The initial rule base

Each rule corresponds to one precedent referential value. The belief degrees show how the function's output at that point is distributed across the consequent referential values. This is analogous to Table 1 in Chen et al. (2011).


In [ ]:
rb = model.rule_base
print("Consequent referential values:", crv)
print()
for k in range(rb.n_rules):
    x_k = prv[0][k]
    y_k = x_k * np.sin(x_k**2)
    beliefs = rb.belief_degrees[k]
    print(f"Rule {k + 1}: x={x_k:.1f}  f(x)={y_k:+.3f}  beliefs={np.round(beliefs, 3)}")

In [ ]:
# Predict on a dense grid (untrained model)
X_eval = np.linspace(0, 3, 500).reshape(-1, 1)
y_true = f(X_eval[:, 0])
y_untrained = model.predict_values(X_eval)

plt.figure(figsize=(8, 4))
plt.plot(x_dense, f(x_dense), "k-", linewidth=2, label="True f(x)")
plt.plot(X_eval[:, 0], y_untrained, "b--", linewidth=1.5, label="Untrained BRB")
plt.scatter(
    prv[0], [f(x) for x in prv[0]], c="red", s=60, zorder=5, label="Referential values"
)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Untrained BRB prediction")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Training

The `fit()` method optimizes:

- **Belief degrees** (how each rule distributes belief across consequent values)
- **Rule weights** (relative importance of each rule)
- **Attribute weights** (importance of each input attribute per rule)
- **Referential value positions** (the anchor points themselves can shift)

All optimization is subject to constraints (Chen et al. 2011, Eqs. 6a-6e): belief rows sum to 1, rule weights sum to 1, attribute weights are non-negative, and referential values remain sorted. The objective is MSE (Eq. 7b in Chen et al. 2011).

**Non-convex optimization.** BRB training has multiple local minima. Using `n_restarts > 1` runs the optimizer from several randomly perturbed starting points and keeps the best result, which dramatically improves solution quality.

**Available methods:** `SLSQP` (default), `trust-constr`, `DE` (differential evolution), `DE+SLSQP` (global search then local polish), `ipopt` (with Pyomo). See the README for a comparison table.


In [ ]:
# Generate training data (1000 evenly spaced points, as in the paper)
X_train = np.linspace(0, 3, 1000).reshape(-1, 1)
y_train = X_train[:, 0] * np.sin(X_train[:, 0] ** 2)

# Train with multiple restarts for reliable results.
# BRB training has multiple local minima; n_restarts runs the
# optimizer from perturbed starting points and keeps the best.
# Alternative: method='DE+SLSQP' for global search without restarts.
model.fit(X_train, y_train, fix_endpoints=True, n_restarts=5, method="SLSQP", verbose=True)

# Predict (trained model)
y_trained = model.predict_values(X_eval)

# Compare
mse_untrained = np.mean((y_true - y_untrained) ** 2)
mse_trained = np.mean((y_true - y_trained) ** 2)
print(f"Untrained MSE: {mse_untrained:.4f}")
print(f"Trained MSE:   {mse_trained:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_dense, f(x_dense), "k-", linewidth=2, label="True f(x)")
plt.plot(X_eval[:, 0], y_untrained, "b--", linewidth=1, alpha=0.5, label="Untrained")
plt.plot(X_eval[:, 0], y_trained, "r-", linewidth=1.5, label="Trained")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Trained vs untrained BRB")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Inspecting the inference trace

Every `predict()` call returns an `InferenceResult` containing the full trace of the inference pipeline: input belief distributions, activation weights, combined belief degrees, and scalar outputs. This is what makes BRB systems **explainable**.


In [ ]:
# Predict at a single point
X_point = np.array([[1.5]])
result = model.predict(X_point)

print(f"Input: x = {X_point[0, 0]}")
print(f"True value: f(x) = {f(X_point[0, 0]):.4f}")
print(f"Predicted:  {result.output[0]:.4f}")
print()
print("Activation weights (per rule):")
for k in range(model.rule_base.n_rules):
    w = result.activation_weights[0, k]
    if w > 0.01:
        print(f"  Rule {k + 1} (x={prv[0][k]:.1f}): {w:.4f}")
print()
print("Combined belief degrees:")
for i, d in enumerate(crv):
    print(f"  D={d:+.1f}: {result.combined_belief_degrees[0, i]:.4f}")
print()
print(f"Dominant rules: {result.dominant_rules(top_k=3)[0]}")

## Summary

This notebook demonstrated the basic BRB workflow:

1. Define referential values for inputs and outputs
2. Construct a model (optionally initialized from a function or expert knowledge)
3. Train the model from data
4. Predict and inspect the full inference trace

**Next:** See `02_multi_attribute.ipynb` for multi-input examples, and `03_expert_knowledge.ipynb` for building models from domain expertise.
